# 24b — label-aware horizontal flip at p = 0.25

Baseline: rung 21 arm A (`lr=1e-4`). The one variable is the deterministic per-row horizontal-flip probability. Training images only; evaluation remains untouched.

Sibling rung: use `FLIP_PROB = 0.50` and a new `RUN_NAME` only after 24b has been evaluated.

In [ ]:
SMOKE = True
FLIP_PROB = 0.25
SEED = 42
RUN_NAME = '24_flip_p25_v1'

In [ ]:
import os, sys
from pathlib import Path

EXP_DIR = Path.cwd()
REPO = EXP_DIR
while REPO != REPO.parent and not (REPO / 'src').is_dir():
    REPO = REPO.parent
if EXP_DIR.name != '24-geometric-aug':
    EXP_DIR = REPO / 'experiments' / '24-geometric-aug'
for path in (
    EXP_DIR / '_models', REPO / 'src', REPO / 'vendor' / 'orena-focus' / 'src',
    REPO / 'experiments' / '21-recipe-sweep' / '_models',
    REPO / 'experiments' / '18-count-aug' / '_models',
    REPO / 'experiments' / '06-vit-lora' / '_models',
    REPO / 'experiments' / '02-lora-sft' / '_models',
):
    if path.is_dir():
        sys.path.insert(0, str(path))
env_bin = str(Path(sys.executable).parent)
if env_bin not in os.environ.get('PATH', '').split(os.pathsep):
    os.environ['PATH'] = env_bin + os.pathsep + os.environ.get('PATH', '')

from horizontal_flip import HorizontalFlipConfig, HorizontalFlipPolicy, gate_single_variable, main

effective_run = '24_flip_smoke' if SMOKE else RUN_NAME
cfg = HorizontalFlipConfig(
    exp_dir=EXP_DIR, run_name=effective_run,
    model_path=Path('/workspace/models/qwen3-vl-8b'),
    control_train_jsonl=Path('/workspace/repo/experiments/18-count-aug/runs/18_count_aug_v1/train.jsonl'),
    control_sha256='180e28f0325674197d52706beeabd846851bdd2875264b5c3505e0debfbd8e8b',
    learning_rate=1e-4, seed=SEED, smoke=SMOKE, smoke_max_steps=4,
    flip=HorizontalFlipPolicy(probability=FLIP_PROB, seed=SEED),
)
cfg.run_dir.mkdir(parents=True, exist_ok=True)
print({'run': cfg.run_name, 'smoke': cfg.smoke, 'flip_probability': cfg.flip.probability, 'baseline': '21_lr_1e4_v1'})

In [ ]:
# G-OFF: no policy means the exact rung-21 control dataset; G-D: policy-on changes only --dataset.
from dataclasses import replace
print(gate_single_variable(replace(cfg, flip=None)))
main(cfg, stage='export')
print(gate_single_variable(cfg))

import pandas as pd
manifest = pd.read_csv(cfg.flip_manifest_csv)
print(manifest.groupby(['flipped', 'rule']).size())
control_rows = sum(
    1 for _ in cfg.control_train_jsonl.open(encoding="utf-8")
)
assert len(manifest) == (cfg.flip_smoke_rows if SMOKE else control_rows)
assert manifest.flipped.any(), 'smoke did not exercise the flip path'

In [ ]:
# Training is only authorised after the export/gates above pass.
main(cfg, stage='train')

## Before a full run

Run the smoke, then obtain the required independent GO/NO-GO review. For the full p=0.25 run, execute this notebook with `SMOKE=False`; do not change any other config. Score every epoch canonically against rung 21 arm A.